# 📦 FitMind — GGUF Conversion (dependency-safe)

**Pipeline:** Merge → Convert → Quantize → Compare

**KEY RULE baked in:** numpy is pinned to 2.x BEFORE any import, and gguf is
installed only at the convert step (it conflicts with numpy if installed early).
So **no runtime restart is needed** — just run top to bottom.

**Before running:** Runtime → T4 GPU. Upload `fitmind-lora-final.zip` + `test_chat.jsonl`.

## Cell 1 — Install (peft only) + pin numpy. DO NOT import anything here.
We install before importing, so the first import below gets the right numpy.

In [ ]:
# peft for the merge. Force numpy 2.x (Colab's torch/transformers need it).
# NO gguf / sentencepiece here -- they downgrade numpy and break transformers.
# Remove torchao -- new peft demands torchao>=0.16 but Colab ships an old one,
# and we don't use torchao for a standard LoRA merge.
!pip install -q peft
!pip uninstall -y -q torchao
!pip install -q --force-reinstall --no-deps "numpy==2.0.2"
print('Installed peft + numpy 2.0.2, removed torchao. No restart needed.')

## Cell 2 — Verify imports (numpy must be 2.x)

In [ ]:
import numpy
print('numpy:', numpy.__version__)            # must start with 2.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set T4!')
print('ALL IMPORTS OK')

## Cell 3 — Clone + build llama.cpp (only llama-quantize, ~4 min)

In [ ]:
import os
if not os.path.exists('llama.cpp'):
    !git clone --depth 1 https://github.com/ggerganov/llama.cpp
!cd llama.cpp && cmake -B build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF 2>&1 | tail -2
!cd llama.cpp && cmake --build build --config Release -j4 --target llama-quantize 2>&1 | tail -5
!ls -la llama.cpp/build/bin/llama-quantize
print('Build done')

## Cell 4 — Upload + unzip adapter
Upload `fitmind-lora-final.zip` (or it's already in Files).

In [ ]:
import os
if not os.path.exists('fitmind-lora-final'):
    if not os.path.exists('fitmind-lora-final.zip'):
        from google.colab import files; files.upload()
    !unzip -q fitmind-lora-final.zip
print('Adapter files:', os.listdir('fitmind-lora-final'))

## Cell 5 — STEP 1: Merge adapter into base (LoRA → full FP16)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

BASE = 'Qwen/Qwen3-1.7B'
base = AutoModelForCausalLM.from_pretrained(
    BASE, torch_dtype=torch.float16, device_map='cpu', low_cpu_mem_usage=True)
model = PeftModel.from_pretrained(base, 'fitmind-lora-final')
model = model.merge_and_unload()                      # the merge
model.save_pretrained('fitmind-merged', safe_serialization=True)
AutoTokenizer.from_pretrained('fitmind-lora-final').save_pretrained('fitmind-merged')
print('Merged FP16 model saved.')
!du -sh fitmind-merged

## Cell 6 — STEP 2: Install gguf + Convert HF → GGUF (FP16)
gguf is installed HERE (after merge). We re-pin numpy 2.x so the converter
subprocess (which imports torch) still works.

In [ ]:
!pip install -q gguf
!pip install -q --force-reinstall --no-deps "numpy==2.0.2"
!python llama.cpp/convert_hf_to_gguf.py fitmind-merged \
  --outfile fitmind-f16.gguf --outtype f16
!ls -lh fitmind-f16.gguf

## Cell 7 — STEP 3: Quantize to Q4_K_M, Q5_K_M, Q6_K

In [ ]:
Q = 'llama.cpp/build/bin/llama-quantize'
for qtype in ['Q4_K_M', 'Q5_K_M', 'Q6_K']:
    print('=== ' + qtype + ' ===')
    !{Q} fitmind-f16.gguf fitmind-{qtype}.gguf {qtype} 2>&1 | tail -2
print('All quantized.')

## Cell 8 — Compare file sizes

In [ ]:
import os
print('SIZE COMPARISON:')
for f in ['fitmind-f16.gguf','fitmind-Q6_K.gguf','fitmind-Q5_K_M.gguf','fitmind-Q4_K_M.gguf']:
    if os.path.exists(f):
        print('  {:24s} {:6.0f} MB'.format(f, os.path.getsize(f)/1e6))

## Cell 9 — Quality check: quick test each quant
Installs llama-cpp-python (CPU) and runs one prompt per quant to confirm
they still produce valid output after quantization.

In [ ]:
!pip install -q llama-cpp-python
from llama_cpp import Llama

SYS = ('You are FitMind, an offline calorie tracker for Indian gym users. '
       'Extract food items, estimate calories as [min,max], sum macros, add to '
       'the running total from context, compute budget left. Reply in the same '
       'language. Answer with a single JSON object.')
USER = ('Daily budget: 2000 cal\nEaten today: nothing yet\nTotal so far: 0 cal\n\n'
        'User (hinglish): abhi maine 3 roti aur ek bowl rajma khaya')

for q in ['Q6_K','Q5_K_M','Q4_K_M']:
    p = 'fitmind-' + q + '.gguf'
    if not os.path.exists(p):
        continue
    llm = Llama(model_path=p, n_ctx=1024, verbose=False)
    out = llm.create_chat_completion(
        messages=[{'role':'system','content':SYS},{'role':'user','content':USER}],
        max_tokens=200, temperature=0)
    print('\n===== ' + q + ' =====')
    print(out['choices'][0]['message']['content'][:300])
    del llm

## Cell 10 — Download chosen GGUF (default Q4_K_M)

In [ ]:
from google.colab import files
CHOSEN = 'fitmind-Q4_K_M.gguf'   # change to Q5_K_M / Q6_K if Cell 9 shows Q4 is worse
files.download(CHOSEN)